In [1]:
%pip install azure-eventhub

StatementMeta(, fd0aaf5c-523b-4690-a52a-bd980e79ae53, 7, Finished, Available, Finished)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 5.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 15.6 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.5.0
    Not uninstalling typing-extensions at /home/trusted-service-user/cluster-env/trident_env/lib/python3.10/site-packages, outside environment /nfs4/pyenv-e2c180be-0adc-402e-9d18-9672e2ce452e
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 2.0.0 requires sentencepiece, which is not installed.
sentence-transformers 2.0.0 requires torchvision, which is not installed.
dash 2.14.0 requires Flask<2.3.0,>=1.0.4, but you have flask 3.0.0 which is incompatible.
dash 2.14.0 requires Werkzeug<2.3.0, but you hav

In [ ]:
import time
import uuid
import json
import random
from datetime import datetime
from azure.eventhub import EventHubProducerClient, EventData


# -------------------------------------
# FABRIC EVENTSTREAM CONNECTION
# -------------------------------------
# Get this from your Event Stream's Custom App source → Event Hub tab → SAS Key Authentication
EVENT_HUB_CONNECTION_STRING = "Endpoint=sb://xxx.servicebus.windows.net/;SharedAccessKeyName=xxx;SharedAccessKey=xxx;EntityPath=xxx"

# -------------------------------------
# ENERGY SOURCE TYPES
# -------------------------------------
ENERGY_SOURCES = [
    {"type": "Solar", "unit": "kWh", "base_capacity": 500, "variance": 0.4},
    {"type": "Wind", "unit": "kWh", "base_capacity": 800, "variance": 0.6},
    {"type": "Marine", "unit": "kWh", "base_capacity": 300, "variance": 0.3},
    {"type": "Hydroelectric", "unit": "kWh", "base_capacity": 1000, "variance": 0.2},
]

# Anomaly tracking - triggers every hour for SUB_007
ANOMALY_SUBSTATION = "SUB_007"
last_anomaly_hour = -1


# -------------------------------------
# ENERGY SUBSTATION SIMULATOR CLASS
# -------------------------------------
class EnergySubstationSimulator:

    def __init__(self, substation_id: str, substation_name: str, energy_type: str, producer_client):
        self.substation_id = substation_id
        self.substation_name = substation_name
        self.energy_type = energy_type
        self.producer_client = producer_client
        
        # Get base capacity and variance for this energy type
        source_config = next((s for s in ENERGY_SOURCES if s["type"] == energy_type), ENERGY_SOURCES[0])
        self.base_capacity = source_config["base_capacity"]
        self.variance = source_config["variance"]

    def generate_energy_reading(self):
        """Generate a single energy generation reading with realistic fluctuations."""
        global last_anomaly_hour
        
        # Simulate time-of-day variations (hour of day affects generation)
        hour = datetime.now().hour
        
        # Apply time-based modifiers
        if self.energy_type == "Solar":
            # Solar peaks during midday (10-16h)
            time_factor = max(0, (1 - abs(hour - 13) / 8)) if 6 <= hour <= 20 else 0
        elif self.energy_type == "Wind":
            # Wind is more variable but stronger at night/morning
            time_factor = 0.7 + 0.3 * random.random()
        elif self.energy_type == "Marine":
            # Marine (tidal) has periodic patterns
            time_factor = 0.5 + 0.5 * abs((hour % 6) - 3) / 3
        else:  # Hydroelectric
            # Hydroelectric is most stable
            time_factor = 0.9 + 0.1 * random.random()
        
        # Calculate generated power with variance
        generated_kwh = round(
            self.base_capacity * time_factor * (1 + random.uniform(-self.variance, self.variance)),
            2
        )
        
        # Ensure non-negative values
        generated_kwh = max(0, generated_kwh)
        
        # ANOMALY DETECTION: Inject anomaly every hour for specific substation
        is_anomaly = False
        if self.substation_id == ANOMALY_SUBSTATION and last_anomaly_hour != hour:
            # Trigger anomaly once per hour
            last_anomaly_hour = hour
            is_anomaly = True
            
            # Apply anomaly: random spike (3x) or drop (0.1x)
            anomaly_type = random.choice(["spike", "drop"])
            if anomaly_type == "spike":
                generated_kwh = generated_kwh * random.uniform(2.5, 4.0)
                print(f"⚠️ ANOMALY TRIGGERED: {self.substation_id} - POWER SPIKE")
            else:
                generated_kwh = generated_kwh * random.uniform(0.05, 0.15)
                print(f"⚠️ ANOMALY TRIGGERED: {self.substation_id} - POWER DROP")
            
            generated_kwh = round(generated_kwh, 2)
        
        # Calculate efficiency (85-98%)
        efficiency = round(random.uniform(0.85, 0.98), 3)
        
        # Temperature affects performance (15-35°C typical range)
        ambient_temp = round(random.uniform(15, 35), 1)
        
        return {
            "reading_id": str(uuid.uuid4()),
            "timestamp": datetime.now().isoformat(),
            "substation_id": self.substation_id,
            "substation_name": self.substation_name,
            "energy_type": self.energy_type,
            "generated_kwh": generated_kwh,
            "efficiency_percent": round(efficiency * 100, 1),
            "ambient_temperature_c": ambient_temp,
            "status": "operational" if generated_kwh > 0 else "idle",
            "grid_voltage_kv": round(random.uniform(130, 145), 1),
            "is_anomaly": is_anomaly
        }

    def send_to_fabric(self, data):
        """Send event to Microsoft Fabric Eventstream using Event Hub protocol."""
        try:
            # Create event batch
            event_data_batch = self.producer_client.create_batch()
            
            # Add event to batch (convert to JSON string)
            event_data_batch.add(EventData(json.dumps(data)))
            
            # Send the batch
            self.producer_client.send_batch(event_data_batch)
            
            status = "🚨 ANOMALY" if data.get("is_anomaly") else "✔"
            print(f"{status} Energy reading sent to Fabric Event Stream")

        except Exception as e:
            print(f"❌ Failed to send event: {e}")

    def process_reading(self):
        reading = self.generate_energy_reading()
        print(json.dumps(reading, indent=2))
        self.send_to_fabric(reading)


# -------------------------------------
# MAIN LOOP – REALTIME SIMULATION
# -------------------------------------
def main():
    
    # Create Event Hub producer client
    try:
        producer = EventHubProducerClient.from_connection_string(
            conn_str=EVENT_HUB_CONNECTION_STRING
        )
        print("✅ Connected to Fabric Event Stream")
    except Exception as e:
        print(f"❌ Failed to connect to Event Stream: {e}")
        print("Please check your connection string!")
        return

    # Initialize Energy Substations - 20 total
    SUBSTATION_LIST = [
        EnergySubstationSimulator("SUB_001", "Madrid Solar Park", "Solar", producer),
        EnergySubstationSimulator("SUB_002", "Barcelona Wind Farm", "Wind", producer),
        EnergySubstationSimulator("SUB_003", "Valencia Marine Station", "Marine", producer),
        EnergySubstationSimulator("SUB_004", "Zaragoza Solar Array", "Solar", producer),
        EnergySubstationSimulator("SUB_005", "Seville Hydro Plant", "Hydroelectric", producer),
        EnergySubstationSimulator("SUB_006", "Bilbao Wind Turbines", "Wind", producer),
        EnergySubstationSimulator("SUB_007", "Almeria Solar Complex", "Solar", producer),  # ANOMALY SUBSTATION
        EnergySubstationSimulator("SUB_008", "Galicia Marine Array", "Marine", producer),
        EnergySubstationSimulator("SUB_009", "Malaga Solar Farm", "Solar", producer),
        EnergySubstationSimulator("SUB_010", "Murcia Wind Park", "Wind", producer),
        EnergySubstationSimulator("SUB_011", "Toledo Hydro Station", "Hydroelectric", producer),
        EnergySubstationSimulator("SUB_012", "Granada Solar Plant", "Solar", producer),
        EnergySubstationSimulator("SUB_013", "Asturias Wind Complex", "Wind", producer),
        EnergySubstationSimulator("SUB_014", "Tarragona Marine Station", "Marine", producer),
        EnergySubstationSimulator("SUB_015", "Salamanca Solar Array", "Solar", producer),
        EnergySubstationSimulator("SUB_016", "Navarra Wind Farm", "Wind", producer),
        EnergySubstationSimulator("SUB_017", "Extremadura Hydro Plant", "Hydroelectric", producer),
        EnergySubstationSimulator("SUB_018", "Cantabria Marine Array", "Marine", producer),
        EnergySubstationSimulator("SUB_019", "Cordoba Solar Complex", "Solar", producer),
        EnergySubstationSimulator("SUB_020", "Leon Wind Turbines", "Wind", producer),
    ]

    print("🚀 Real-Time Energy Generation → Fabric Eventstream Simulator Started")
    print(f"⚠️  Anomaly Detection: {ANOMALY_SUBSTATION} (Almeria Solar Complex) will have anomalies every hour")
    print("-----------------------------------------------------------------------")

    try:
        while True:
            substation = random.choice(SUBSTATION_LIST)
            substation.process_reading()
            time.sleep(random.uniform(3, 10))   # simulate real-time energy readings
    
    except KeyboardInterrupt:
        print("\n⏹ Stopping simulator...")
    
    finally:
        # Close the producer client
        producer.close()
        print("✅ Connection closed")


if __name__ == "__main__":
    main()

StatementMeta(, fd0aaf5c-523b-4690-a52a-bd980e79ae53, 9, Submitted, Running, Running)

✅ Connected to Fabric Event Stream
🚀 Real-Time Energy Generation → Fabric Eventstream Simulator Started
⚠️  Anomaly Detection: SUB_007 (Almeria Solar Complex) will have anomalies every hour
-----------------------------------------------------------------------
{
  "reading_id": "9df45ec4-f463-4d49-b821-d2d353e679d4",
  "timestamp": "2025-12-23T16:07:49.726928",
  "substation_id": "SUB_009",
  "substation_name": "Malaga Solar Farm",
  "energy_type": "Solar",
  "generated_kwh": 243.29,
  "efficiency_percent": 85.2,
  "ambient_temperature_c": 23.4,
  "status": "operational",
  "grid_voltage_kv": 139.2,
  "is_anomaly": false
}
✔ Energy reading sent to Fabric Event Stream
{
  "reading_id": "d607ee3c-838f-4c25-82a8-2a1caafc18d8",
  "timestamp": "2025-12-23T16:07:59.070095",
  "substation_id": "SUB_001",
  "substation_name": "Madrid Solar Park",
  "energy_type": "Solar",
  "generated_kwh": 374.05,
  "efficiency_percent": 92.5,
  "ambient_temperature_c": 24.4,
  "status": "operational",
  "gr

In [2]:
import pandas as pd

# Substation Dimension Table with Real Locations in Spain - 20 substations
substations_data = [
    {
        "substation_id": "SUB_001",
        "substation_name": "Madrid Solar Park",
        "energy_type": "Solar",
        "city": "Madrid",
        "region": "Community of Madrid",
        "country": "Spain",
        "postal_code": "28220",
        "address": "Carretera de Colmenar Viejo, km 15",
        "latitude": 40.5423,
        "longitude": -3.6345,
        "installation_date": "2020-05-15",
        "capacity_mw": 50.0,
        "number_of_panels_turbines": 150000,
        "area_hectares": 85.0,
        "average_annual_output_gwh": 75.5
    },
    {
        "substation_id": "SUB_002",
        "substation_name": "Barcelona Wind Farm",
        "energy_type": "Wind",
        "city": "Barcelona",
        "region": "Catalonia",
        "country": "Spain",
        "postal_code": "08830",
        "address": "Collserola Natural Park",
        "latitude": 41.4231,
        "longitude": 2.1123,
        "installation_date": "2019-03-22",
        "capacity_mw": 80.0,
        "number_of_panels_turbines": 40,
        "area_hectares": 120.0,
        "average_annual_output_gwh": 168.0
    },
    {
        "substation_id": "SUB_003",
        "substation_name": "Valencia Marine Station",
        "energy_type": "Marine",
        "city": "Valencia",
        "region": "Valencian Community",
        "country": "Spain",
        "postal_code": "46024",
        "address": "Port of Valencia - Marine Energy Complex",
        "latitude": 39.4561,
        "longitude": -0.3318,
        "installation_date": "2021-09-10",
        "capacity_mw": 30.0,
        "number_of_panels_turbines": 15,
        "area_hectares": 5.0,
        "average_annual_output_gwh": 52.6
    },
    {
        "substation_id": "SUB_004",
        "substation_name": "Zaragoza Solar Array",
        "energy_type": "Solar",
        "city": "Zaragoza",
        "region": "Aragon",
        "country": "Spain",
        "postal_code": "50720",
        "address": "Polígono Industrial El Pradillo",
        "latitude": 41.7342,
        "longitude": -0.9534,
        "installation_date": "2018-11-28",
        "capacity_mw": 65.0,
        "number_of_panels_turbines": 195000,
        "area_hectares": 110.0,
        "average_annual_output_gwh": 98.8
    },
    {
        "substation_id": "SUB_005",
        "substation_name": "Seville Hydro Plant",
        "energy_type": "Hydroelectric",
        "city": "Seville",
        "region": "Andalusia",
        "country": "Spain",
        "postal_code": "41920",
        "address": "Embalse de la Minilla",
        "latitude": 37.5443,
        "longitude": -5.7234,
        "installation_date": "2017-06-12",
        "capacity_mw": 100.0,
        "number_of_panels_turbines": 4,
        "area_hectares": 250.0,
        "average_annual_output_gwh": 245.0
    },
    {
        "substation_id": "SUB_006",
        "substation_name": "Bilbao Wind Turbines",
        "energy_type": "Wind",
        "city": "Bilbao",
        "region": "Basque Country",
        "country": "Spain",
        "postal_code": "48920",
        "address": "Monte Oiz Wind Complex",
        "latitude": 43.3213,
        "longitude": -2.6534,
        "installation_date": "2019-08-05",
        "capacity_mw": 72.0,
        "number_of_panels_turbines": 36,
        "area_hectares": 95.0,
        "average_annual_output_gwh": 151.2
    },
    {
        "substation_id": "SUB_007",
        "substation_name": "Almeria Solar Complex",
        "energy_type": "Solar",
        "city": "Almeria",
        "region": "Andalusia",
        "country": "Spain",
        "postal_code": "04720",
        "address": "Tabernas Desert Solar Park",
        "latitude": 37.0567,
        "longitude": -2.3521,
        "installation_date": "2020-01-20",
        "capacity_mw": 90.0,
        "number_of_panels_turbines": 270000,
        "area_hectares": 150.0,
        "average_annual_output_gwh": 145.8
    },
    {
        "substation_id": "SUB_008",
        "substation_name": "Galicia Marine Array",
        "energy_type": "Marine",
        "city": "A Coruña",
        "region": "Galicia",
        "country": "Spain",
        "postal_code": "15002",
        "address": "Atlantic Coast Wave Energy Site",
        "latitude": 43.3623,
        "longitude": -8.4115,
        "installation_date": "2022-03-15",
        "capacity_mw": 25.0,
        "number_of_panels_turbines": 12,
        "area_hectares": 3.5,
        "average_annual_output_gwh": 43.8
    },
    {
        "substation_id": "SUB_009",
        "substation_name": "Malaga Solar Farm",
        "energy_type": "Solar",
        "city": "Malaga",
        "region": "Andalusia",
        "country": "Spain",
        "postal_code": "29590",
        "address": "Parque Tecnológico de Andalucía",
        "latitude": 36.6885,
        "longitude": -4.4936,
        "installation_date": "2021-07-18",
        "capacity_mw": 55.0,
        "number_of_panels_turbines": 165000,
        "area_hectares": 92.0,
        "average_annual_output_gwh": 88.2
    },
    {
        "substation_id": "SUB_010",
        "substation_name": "Murcia Wind Park",
        "energy_type": "Wind",
        "city": "Murcia",
        "region": "Region of Murcia",
        "country": "Spain",
        "postal_code": "30530",
        "address": "Sierra de la Pila",
        "latitude": 38.1623,
        "longitude": -1.2345,
        "installation_date": "2020-11-03",
        "capacity_mw": 68.0,
        "number_of_panels_turbines": 34,
        "area_hectares": 105.0,
        "average_annual_output_gwh": 142.8
    },
    {
        "substation_id": "SUB_011",
        "substation_name": "Toledo Hydro Station",
        "energy_type": "Hydroelectric",
        "city": "Toledo",
        "region": "Castilla-La Mancha",
        "country": "Spain",
        "postal_code": "45600",
        "address": "Embalse de Castrejón",
        "latitude": 39.7834,
        "longitude": -4.3521,
        "installation_date": "2018-04-25",
        "capacity_mw": 85.0,
        "number_of_panels_turbines": 3,
        "area_hectares": 180.0,
        "average_annual_output_gwh": 208.5
    },
    {
        "substation_id": "SUB_012",
        "substation_name": "Granada Solar Plant",
        "energy_type": "Solar",
        "city": "Granada",
        "region": "Andalusia",
        "country": "Spain",
        "postal_code": "18320",
        "address": "Valle de Lecrín Solar Complex",
        "latitude": 36.9234,
        "longitude": -3.5678,
        "installation_date": "2019-09-12",
        "capacity_mw": 48.0,
        "number_of_panels_turbines": 144000,
        "area_hectares": 80.0,
        "average_annual_output_gwh": 76.8
    },
    {
        "substation_id": "SUB_013",
        "substation_name": "Asturias Wind Complex",
        "energy_type": "Wind",
        "city": "Oviedo",
        "region": "Asturias",
        "country": "Spain",
        "postal_code": "33840",
        "address": "Cordillera Cantábrica Wind Farm",
        "latitude": 43.2456,
        "longitude": -5.9123,
        "installation_date": "2020-02-28",
        "capacity_mw": 76.0,
        "number_of_panels_turbines": 38,
        "area_hectares": 115.0,
        "average_annual_output_gwh": 159.6
    },
    {
        "substation_id": "SUB_014",
        "substation_name": "Tarragona Marine Station",
        "energy_type": "Marine",
        "city": "Tarragona",
        "region": "Catalonia",
        "country": "Spain",
        "postal_code": "43004",
        "address": "Port of Tarragona - Wave Energy Hub",
        "latitude": 41.1167,
        "longitude": 1.2444,
        "installation_date": "2022-06-20",
        "capacity_mw": 28.0,
        "number_of_panels_turbines": 14,
        "area_hectares": 4.2,
        "average_annual_output_gwh": 49.0
    },
    {
        "substation_id": "SUB_015",
        "substation_name": "Salamanca Solar Array",
        "energy_type": "Solar",
        "city": "Salamanca",
        "region": "Castilla y León",
        "country": "Spain",
        "postal_code": "37185",
        "address": "Villamayor Solar Park",
        "latitude": 40.9943,
        "longitude": -5.6789,
        "installation_date": "2021-03-10",
        "capacity_mw": 52.0,
        "number_of_panels_turbines": 156000,
        "area_hectares": 88.0,
        "average_annual_output_gwh": 83.2
    },
    {
        "substation_id": "SUB_016",
        "substation_name": "Navarra Wind Farm",
        "energy_type": "Wind",
        "city": "Pamplona",
        "region": "Navarre",
        "country": "Spain",
        "postal_code": "31191",
        "address": "Bardenas Reales Wind Complex",
        "latitude": 42.3667,
        "longitude": -1.5234,
        "installation_date": "2019-12-15",
        "capacity_mw": 82.0,
        "number_of_panels_turbines": 41,
        "area_hectares": 125.0,
        "average_annual_output_gwh": 172.2
    },
    {
        "substation_id": "SUB_017",
        "substation_name": "Extremadura Hydro Plant",
        "energy_type": "Hydroelectric",
        "city": "Cáceres",
        "region": "Extremadura",
        "country": "Spain",
        "postal_code": "10600",
        "address": "Embalse de Alcántara",
        "latitude": 39.7223,
        "longitude": -6.5234,
        "installation_date": "2018-08-30",
        "capacity_mw": 95.0,
        "number_of_panels_turbines": 4,
        "area_hectares": 220.0,
        "average_annual_output_gwh": 233.0
    },
    {
        "substation_id": "SUB_018",
        "substation_name": "Cantabria Marine Array",
        "energy_type": "Marine",
        "city": "Santander",
        "region": "Cantabria",
        "country": "Spain",
        "postal_code": "39004",
        "address": "Bay of Biscay Energy Station",
        "latitude": 43.4623,
        "longitude": -3.8099,
        "installation_date": "2021-11-08",
        "capacity_mw": 32.0,
        "number_of_panels_turbines": 16,
        "area_hectares": 5.5,
        "average_annual_output_gwh": 56.0
    },
    {
        "substation_id": "SUB_019",
        "substation_name": "Cordoba Solar Complex",
        "energy_type": "Solar",
        "city": "Cordoba",
        "region": "Andalusia",
        "country": "Spain",
        "postal_code": "14440",
        "address": "Campiña Solar Park",
        "latitude": 37.8845,
        "longitude": -4.7796,
        "installation_date": "2020-09-05",
        "capacity_mw": 58.0,
        "number_of_panels_turbines": 174000,
        "area_hectares": 97.0,
        "average_annual_output_gwh": 92.8
    },
    {
        "substation_id": "SUB_020",
        "substation_name": "Leon Wind Turbines",
        "energy_type": "Wind",
        "city": "León",
        "region": "Castilla y León",
        "country": "Spain",
        "postal_code": "24750",
        "address": "Montaña Leonesa Wind Park",
        "latitude": 42.8234,
        "longitude": -5.5678,
        "installation_date": "2020-05-22",
        "capacity_mw": 70.0,
        "number_of_panels_turbines": 35,
        "area_hectares": 110.0,
        "average_annual_output_gwh": 147.0
    }
]

# Create DataFrame
df_substations = pd.DataFrame(substations_data)

# Display the table
print("=" * 120)
print("ENERGY SUBSTATION DIMENSION TABLE - 20 SUBSTATIONS")
print("=" * 120)
print(df_substations.to_string(index=False))
print("=" * 120)
print(f"\nTotal Substations: {len(df_substations)}")
print(f"Total Capacity: {df_substations['capacity_mw'].sum():.1f} MW")
print(f"Total Annual Output: {df_substations['average_annual_output_gwh'].sum():.1f} GWh")
print(f"Average Capacity per Substation: {df_substations['capacity_mw'].mean():.1f} MW")

# Export to CSV (optional)
df_substations.to_csv('substation_dimension.csv', index=False)
print("\n✅ Exported to 'substation_dimension.csv'")

# Save to Lakehouse as Delta Table
try:
    from pyspark.sql import SparkSession
    
    # Get or create Spark session
    spark = SparkSession.builder.getOrCreate()
    
    # Convert pandas DataFrame to PySpark DataFrame
    spark_df = spark.createDataFrame(df_substations)
    
    # Write to Lakehouse Tables folder as Delta format
    spark_df.write.format("delta").mode("overwrite").saveAsTable("dim_substation")
    print("✅ Saved to Lakehouse as Delta table: 'dim_substation'")
    
    # Also save as Parquet in Files section (optional)
    spark_df.write.mode("overwrite").parquet("Files/dimensions/dim_substation.parquet")
    print("✅ Saved to Lakehouse Files: 'Files/dimensions/dim_substation.parquet'")
    
except Exception as e:
    print(f"⚠️ Note: To save to Lakehouse, run this in a Fabric notebook attached to a Lakehouse")
    print(f"   Error: {e}")

# Show summary by energy type
print("\n" + "=" * 120)
print("SUMMARY BY ENERGY TYPE")
print("=" * 120)
summary = df_substations.groupby('energy_type').agg({
    'substation_id': 'count',
    'capacity_mw': 'sum',
    'average_annual_output_gwh': 'sum',
    'area_hectares': 'sum'
}).rename(columns={
    'substation_id': 'Number of Substations',
    'capacity_mw': 'Total Capacity (MW)',
    'average_annual_output_gwh': 'Total Annual Output (GWh)',
    'area_hectares': 'Total Area (hectares)'
})
print(summary)

# Show geographic distribution
print("\n" + "=" * 120)
print("GEOGRAPHIC DISTRIBUTION")
print("=" * 120)
geo_summary = df_substations.groupby('region').agg({
    'substation_id': 'count',
    'capacity_mw': 'sum'
}).rename(columns={
    'substation_id': 'Number of Substations',
    'capacity_mw': 'Total Capacity (MW)'
}).sort_values('Total Capacity (MW)', ascending=False)
print(geo_summary)

# Highlight anomaly substation
print("\n" + "=" * 120)
print("⚠️ ANOMALY MONITORING SUBSTATION")
print("=" * 120)
anomaly_sub = df_substations[df_substations['substation_id'] == 'SUB_007']
print(anomaly_sub.to_string(index=False))
print("\n⚠️ This substation (SUB_007 - Almeria Solar Complex) has hourly anomalies in the real-time stream")

StatementMeta(, 0e4b8856-823e-4f9b-b8c3-9318a2205e9d, 9, Finished, Available, Finished)

ENERGY SUBSTATION DIMENSION TABLE - 20 SUBSTATIONS
substation_id          substation_name   energy_type      city              region country postal_code                                  address  latitude  longitude installation_date  capacity_mw  number_of_panels_turbines  area_hectares  average_annual_output_gwh
      SUB_001        Madrid Solar Park         Solar    Madrid Community of Madrid   Spain       28220       Carretera de Colmenar Viejo, km 15   40.5423    -3.6345        2020-05-15         50.0                     150000           85.0                       75.5
      SUB_002      Barcelona Wind Farm          Wind Barcelona           Catalonia   Spain       08830                  Collserola Natural Park   41.4231     2.1123        2019-03-22         80.0                         40          120.0                      168.0
      SUB_003  Valencia Marine Station        Marine  Valencia Valencian Community   Spain       46024 Port of Valencia - Marine Energy Complex   39.4561 